# 태그 조합 분석

**분석 질문**
- 어떤 태그가 높은 긍정률과 연관되는가?
- 장르 내에서 차별화에 도움이 되는 태그 조합은 무엇인가?
- 출시 전 태그 설정 시 참고할 수 있는 고긍정률 태그는?

In [ ]:
import ast
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

print('라이브러리 로드 완료')

In [2]:
DATA_PATH = Path('../../../data/preprocessed/steam_indie_games.csv')
df_games = pd.read_csv(DATA_PATH)
print(f'데이터 로드 완료: {df_games.shape[0]:,}개 게임')

데이터 로드 완료: 9,169개 게임


In [3]:
def parse_tags(value: str) -> dict[str, int]:
    """JSON 형태의 태그 딕셔너리를 파싱합니다."""
    if pd.isna(value):
        return {}
    try:
        return json.loads(value)
    except (json.JSONDecodeError, TypeError):
        return {}


def top_tags(tag_dict: dict, n: int = 5) -> list[str]:
    """투표수 기준 상위 n개 태그를 반환합니다."""
    return sorted(tag_dict, key=tag_dict.get, reverse=True)[:n]


df_clean = df_games.copy()
for col in ['positive', 'negative', 'total_reviews']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

df_clean = df_clean.dropna(subset=['positive', 'negative', 'total_reviews'])
df_clean = df_clean[df_clean['total_reviews'] > 0].copy()
df_clean['positive_rate'] = df_clean['positive'] / df_clean['total_reviews'] * 100
df_clean['tag_dict'] = df_clean['tags'].apply(parse_tags)
df_clean['top_tags'] = df_clean['tag_dict'].apply(lambda d: top_tags(d, n=5))
df_clean = df_clean[df_clean['top_tags'].map(len) > 0].copy()

print(f'분석 가능 게임 수: {len(df_clean):,}개')

분석 가능 게임 수: 9,169개


## 태그별 평균 긍정률 (상위 30개)

In [4]:
MIN_GAMES_PER_TAG = 30
TOP_N_TAGS = 30

df_tag = df_clean.explode('top_tags').rename(columns={'top_tags': 'tag'})
df_tag['tag'] = df_tag['tag'].astype(str).str.strip()
df_tag = df_tag[df_tag['tag'] != ''].copy()

tag_stats = (
    df_tag
    .groupby('tag')
    .agg(
        game_count=('appid', 'nunique'),
        avg_positive_rate=('positive_rate', 'mean'),
        median_positive_rate=('positive_rate', 'median'),
        median_reviews=('total_reviews', 'median'),
    )
    .reset_index()
)

tag_stats_filtered = (
    tag_stats[tag_stats['game_count'] >= MIN_GAMES_PER_TAG]
    .sort_values('avg_positive_rate', ascending=False)
    .reset_index(drop=True)
)

print(f'게임 수 {MIN_GAMES_PER_TAG}개 이상 태그: {len(tag_stats_filtered)}개')
display(tag_stats_filtered.head(10).round(2))

게임 수 30개 이상 태그: 213개


,tag,game_count,avg_positive_rate,median_positive_rate,median_reviews
0,Sokoban,63,95.74,99.11,23.0
1,Level Editor,35,93.85,96.15,26.0
2,Wholesome,72,92.92,96.57,85.5
3,Logic,125,92.79,96.77,34.0
4,Score Attack,52,92.75,98.10,23.0
5,Fast-Paced,47,92.29,94.44,32.0
6,Minimalist,95,91.99,95.65,27.0
7,Precision Platformer,237,91.48,95.74,27.0
8,Narrative,32,91.48,92.94,112.5
9,Grid-Based Movement,43,91.48,95.65,29.0


In [ ]:
plot_df = tag_stats_filtered.head(TOP_N_TAGS).sort_values('avg_positive_rate')

fig = px.bar(
    plot_df,
    x='avg_positive_rate',
    y='tag',
    orientation='h',
    color='avg_positive_rate',
    color_continuous_scale='Blues',
    text='avg_positive_rate',
    title=f'태그별 평균 긍정률 상위 {TOP_N_TAGS}개 (게임 수 {MIN_GAMES_PER_TAG}개 이상)',
    labels={'avg_positive_rate': '평균 긍정률 (%)', 'tag': '태그'},
    range_x=[0, 100],
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=700)
fig.show()

## 장르별 상위 태그 긍정률 히트맵

In [ ]:
def parse_genres(value: str) -> list[str]:
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except (ValueError, SyntaxError):
        pass
    return [item.strip() for item in str(value).split(',') if item.strip()]


df_clean['genre_list'] = df_clean['genres'].apply(parse_genres)
df_genre_tag = (
    df_clean
    .explode('genre_list')
    .rename(columns={'genre_list': 'genre'})
)
df_genre_tag['genre'] = df_genre_tag['genre'].astype(str).str.strip()
df_genre_tag = df_genre_tag[
    (df_genre_tag['genre'] != '') & (df_genre_tag['genre'] != 'Indie')
].copy()
df_genre_tag = df_genre_tag.explode('top_tags').rename(columns={'top_tags': 'tag'})
df_genre_tag['tag'] = df_genre_tag['tag'].astype(str).str.strip()

top_20_tags = tag_stats_filtered.head(20)['tag'].tolist()
df_genre_tag = df_genre_tag[df_genre_tag['tag'].isin(top_20_tags)].copy()

MIN_CELL = 5
cell_data = (
    df_genre_tag
    .groupby(['genre', 'tag'])
    .agg(avg_positive_rate=('positive_rate', 'mean'), count=('appid', 'nunique'))
    .reset_index()
)
cell_data.loc[cell_data['count'] < MIN_CELL, 'avg_positive_rate'] = float('nan')
pivot = cell_data.pivot(index='genre', columns='tag', values='avg_positive_rate').round(1)

fig = px.imshow(
    pivot,
    text_auto='.1f',
    color_continuous_scale='RdYlGn',
    zmin=70,
    zmax=95,
    title=f'장르 × 태그별 평균 긍정률 (게임 수 {MIN_CELL}개 미만 셀 제외)',
    labels={'x': '태그', 'y': '장르', 'color': '평균 긍정률 (%)'},
    aspect='auto',
)
fig.update_layout(height=400)
fig.show()

## 태그별 게임 수 vs 평균 긍정률 산점도

In [ ]:
plot_scatter = tag_stats_filtered.head(50)
label_threshold = plot_scatter['avg_positive_rate'].quantile(0.8)

fig = px.scatter(
    plot_scatter,
    x='game_count',
    y='avg_positive_rate',
    size='median_reviews',
    color='avg_positive_rate',
    color_continuous_scale='RdYlGn',
    text='tag',
    size_max=40,
    title='태그별 게임 수 vs 평균 긍정률 (원 크기: 중위 리뷰 수)',
    labels={
        'game_count': '게임 수',
        'avg_positive_rate': '평균 긍정률 (%)',
        'median_reviews': '중위 리뷰 수',
    },
    hover_data={'median_reviews': True, 'tag': True},
)
# 상위 긍정률 태그만 레이블 표시
fig.update_traces(
    textposition='top center',
    textfont=dict(size=9),
)
# 하위 태그는 텍스트 숨김
for trace in fig.data:
    pass
fig.update_layout(height=550, yaxis_range=[85, 100], coloraxis_showscale=False)
fig.show()

## 요약 테이블

In [8]:
print('태그별 긍정률 요약 (상위 30개)')
display(
    tag_stats_filtered
    .head(TOP_N_TAGS)[['tag', 'game_count', 'avg_positive_rate', 'median_positive_rate', 'median_reviews']]
    .round(2)
    .reset_index(drop=True)
)

태그별 긍정률 요약 (상위 30개)


,tag,game_count,avg_positive_rate,median_positive_rate,median_reviews
0,Sokoban,63,95.74,99.11,23.0
1,Level Editor,35,93.85,96.15,26.0
2,Wholesome,72,92.92,96.57,85.5
3,Logic,125,92.79,96.77,34.0
4,Score Attack,52,92.75,98.10,23.0
5,Fast-Paced,47,92.29,94.44,32.0
6,Minimalist,95,91.99,95.65,27.0
7,Precision Platformer,237,91.48,95.74,27.0
8,Narrative,32,91.48,92.94,112.5
9,Grid-Based Movement,43,91.48,95.65,29.0
